# train

> Fine-tune the PP-OCRv6 medium recognizer with fastai.


In [ ]:
#| default_exp train


The recognizer is trained as a `TorchRecognizer` under a fastai `Learner`. Its weights start from the official checkpoint, a CTC loss and a character error rate drive the loop, and the result is written as an `.npz` that `compile_recognizer` loads back into MAX.

The charset is fixed. The CTC head has 18710 outputs, one per dictionary character plus the blank, so training can change how the model reads but not which characters it can emit.


In [ ]:
#| hide
from nbdev import show_doc


In [ ]:
#| export
#| notest
from __future__ import annotations

import argparse
import random
from pathlib import Path

import fastai.callback.progress  # noqa: F401  progress bar and logging
import fastai.callback.schedule  # noqa: F401  adds `fine_tune` and `lr_find`
import numpy as np
import torch
from fastai.data.core import DataLoaders
from fastai.data.load import DataLoader
from fastai.learner import Learner, Metric
from fastai.torch_core import params
from torch import nn
from torch.nn.utils.rnn import pad_sequence

from solid_lens.core import repo_root
from solid_lens.ocr import load_charset, load_image, rec_input
from solid_lens.torch_rec import TorchRecognizer, load_recognizer, model_arrays
from solid_lens.weights import save_arrays


In [ ]:
#| export
#| notest
ROOT = repo_root()


## Labels


A dataset is a folder of line images and a label file in PaddleOCR's recognition format: one line per image, the image path, a tab, then the text. Paths are relative to the label file unless `root` says otherwise.


In [ ]:
#| export
#| notest
def read_labels(
    path: Path, # Label file
    root: Path | None = None, # Where the image paths start. Defaults to the label file's folder.
) -> list[tuple[Path, str]]:
    "Read `image<TAB>text` lines."
    path = Path(path)
    root = Path(root) if root else path.parent
    items = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        name, _, text = line.partition("\t")
        items.append((root / name, text))
    return items


Class 0 is the CTC blank, so character `i` of the dictionary is class `i + 1`. This is the same offset `_ctc_decode` removes when it reads a prediction.


In [ ]:
#| export
#| notest
def char_table(charset: list[str]) -> dict[str, int]:
    return {char: index + 1 for index, char in enumerate(charset)}


In [ ]:
#| export
#| notest
def encode(text: str, table: dict[str, int]) -> list[int] | None:
    "Class ids for `text`, or None when it has a character outside the dictionary."
    ids = [table.get(char) for char in text]
    return None if None in ids else ids


CTC needs one timestep per character, plus one more between each pair of equal neighbours so the blank can separate them. The recognizer has 40 timesteps. A label that needs more cannot be learned, and with `zero_infinity` it would only contribute nothing, so those lines are dropped up front and counted.


In [ ]:
#| export
#| notest
def ctc_steps(ids: list[int]) -> int:
    "Timesteps CTC needs to emit `ids`."
    return len(ids) + sum(a == b for a, b in zip(ids, ids[1:]))


In [ ]:
#| export
#| notest
def usable(
    items: list[tuple[Path, str]],
    table: dict[str, int],
    steps: int = 40, # Recognizer timesteps
) -> tuple[list[tuple[Path, list[int]]], int]:
    "Items whose text is in the dictionary and fits in `steps`, and how many were dropped."
    kept = []
    for path, text in items:
        ids = encode(text, table)
        if ids is not None and ctc_steps(ids) <= steps:
            kept.append((path, ids))
    return kept, len(items) - len(kept)


In [ ]:
#| notest
from fastcore.test import test_eq

table = char_table(["a", "b", " "])
test_eq(encode("ab ", table), [1, 2, 3])
test_eq(encode("abz", table), None)
test_eq(ctc_steps([1, 1, 2]), 4)
kept, dropped = usable([(Path("x"), "ab"), (Path("y"), "z"), (Path("z"), "a" * 30)], table, steps=40)
test_eq((len(kept), dropped), (1, 2))


## Data


Each item is read through `rec_input`, the same function inference uses, so a crop is resized, scaled and padded the same way in training and in `read_page`. A batch pads the label ids with 0. The blank is never a label, so 0 also marks where a label ends.


In [ ]:
#| export
#| notest
class LineDataset:
    "A line image as a 3x48x320 tensor, and its class ids."

    def __init__(self, items: list[tuple[Path, list[int]]]) -> None:
        self.items = items

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        path, ids = self.items[index]
        tensor, _ = rec_input(load_image(path))
        return torch.from_numpy(tensor), torch.tensor(ids, dtype=torch.long)


In [ ]:
#| export
#| notest
def collate(batch: list[tuple[torch.Tensor, torch.Tensor]]) -> tuple[torch.Tensor, torch.Tensor]:
    "Stack images, and pad labels with 0."
    images = torch.stack([image for image, _ in batch])
    labels = pad_sequence([ids for _, ids in batch], batch_first=True, padding_value=0)
    return images, labels


In [ ]:
#| export
#| notest
def rec_dataloaders(
    items: list[tuple[Path, str]],
    charset: list[str],
    *,
    bs: int = 16,
    valid_pct: float = 0.1,
    seed: int = 42,
    num_workers: int = 0,
) -> DataLoaders:
    "Train and validation loaders over the items that fit the recognizer."
    kept, dropped = usable(items, char_table(charset))
    if dropped:
        print(f"dropped {dropped} of {len(items)} lines: characters outside the dictionary, or too long for 40 timesteps")
    order = list(range(len(kept)))
    random.Random(seed).shuffle(order)
    n_valid = max(1, int(len(kept) * valid_pct))
    valid = [kept[i] for i in order[:n_valid]]
    train = [kept[i] for i in order[n_valid:]]
    if not train:
        raise ValueError(f"{len(kept)} usable lines is too few to hold out {n_valid} for validation")
    return DataLoaders(
        DataLoader(LineDataset(train), bs=bs, shuffle=True, num_workers=num_workers, create_batch=collate),
        DataLoader(LineDataset(valid), bs=bs, shuffle=False, num_workers=num_workers, create_batch=collate),
    )


## Loss and metric


The model returns logits. CTC takes log-probabilities, time first, and every sample uses all 40 timesteps. The blank is class 0, and label lengths come from the padding.


In [ ]:
#| export
#| notest
class CTCLoss(nn.Module):
    "CTC over the recognizer's timesteps. Targets are padded with 0."

    def __init__(self) -> None:
        super().__init__()
        self.ctc = nn.CTCLoss(blank=0, zero_infinity=True)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        batch, steps, _ = logits.shape
        log_probs = logits.float().log_softmax(dim=-1).permute(1, 0, 2)
        input_lengths = torch.full((batch,), steps, dtype=torch.long)
        return self.ctc(log_probs, targets, input_lengths, (targets > 0).sum(dim=1))


Greedy decoding takes the best class at each step, merges runs, and drops the blank. This is what `_ctc_decode` does at inference, before it maps ids to characters.


In [ ]:
#| export
#| notest
def greedy_ids(logits: torch.Tensor) -> list[list[int]]:
    "Best class ids per sample, with repeats merged and blanks removed."
    out = []
    for row in logits.argmax(dim=-1).tolist():
        ids, previous = [], 0
        for index in row:
            if index != 0 and index != previous:
                ids.append(index)
            previous = index
        out.append(ids)
    return out


In [ ]:
#| export
#| notest
def edit_distance(a: list[int], b: list[int]) -> int:
    "Levenshtein distance."
    row = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        previous, row[0] = row[0], i
        for j, y in enumerate(b, 1):
            previous, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, previous + (x != y))
    return row[-1]


In [ ]:
#| notest
test_eq(edit_distance([1, 2, 3], [1, 3]), 1)
test_eq(edit_distance([], [1, 2]), 2)
test_eq(edit_distance([1, 2], [2, 1]), 2)
steps = torch.zeros(1, 6, 5)
for t, k in enumerate([1, 1, 0, 1, 2, 2]):
    steps[0, t, k] = 1
test_eq(greedy_ids(steps), [[1, 1, 2]])


In [ ]:
#| export
#| notest
class CER(Metric):
    "Character error rate: edits to reach the label, over label characters."

    def reset(self) -> None:
        self.edits = 0
        self.chars = 0

    def accumulate(self, learn) -> None:
        for guess, target in zip(greedy_ids(learn.pred.detach()), learn.y.cpu().tolist()):
            label = [index for index in target if index > 0]
            self.edits += edit_distance(guess, label)
            self.chars += len(label)

    @property
    def value(self) -> float:
        return self.edits / max(1, self.chars)

    @property
    def name(self) -> str:
        return "cer"


## Learner


Three parameter groups let fastai use a different learning rate for each. `fine_tune` first trains the last group (the head) alone, then unfreezes everything with rates that grow from the backbone to the head.


In [ ]:
#| export
#| notest
def rec_splitter(model: TorchRecognizer) -> list[list[nn.Parameter]]:
    "Backbone, neck, head."
    backbone = nn.Sequential(model.stem, model.stage2, model.stage3, model.stage4, model.stage5)
    return [params(backbone), params(model.neck), params(model.head)]


In [ ]:
#| export
#| notest
def rec_learner(model: TorchRecognizer, dls: DataLoaders, **kwargs) -> Learner:
    return Learner(dls, model, loss_func=CTCLoss(), metrics=[CER()], splitter=rec_splitter, **kwargs)


## Saving


The model is saved as the same ordered list of arrays the ONNX reader produces, in an `.npz`. `compile_recognizer(checkpoint=...)` loads it into the MAX graph.


In [ ]:
#| export
#| notest
def save_recognizer(model: TorchRecognizer, path: Path) -> None:
    "Write the weights in checkpoint order. The model is read in eval mode."
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    was_training = model.training
    model.eval()
    save_arrays(model_arrays(model), path)
    model.train(was_training)


## Fine-tune


In [ ]:
#| export
#| notest
def finetune(
    labels: Path, # PaddleOCR recognition label file
    *,
    root: Path | None = None, # Where the label file's image paths start
    init: Path | None = None, # Starting weights. Defaults to the official ONNX.
    out: Path | None = None, # Where to write the `.npz`
    epochs: int = 5, # Epochs with everything unfrozen
    freeze_epochs: int = 1, # Epochs with only the head training first
    bs: int = 16,
    lr: float = 1e-3, # Head rate. The neck gets 1/10 and the backbone 1/100 of it.
    valid_pct: float = 0.1,
    num_workers: int = 0,
    seed: int = 42,
) -> Learner:
    charset = load_charset(ROOT / "models" / "ppocrv6_dict.txt", add_space=True)
    model = load_recognizer(init or ROOT / "models" / "medium_rec.onnx")
    dls = rec_dataloaders(read_labels(labels, root), charset, bs=bs, valid_pct=valid_pct,
                          seed=seed, num_workers=num_workers)
    learn = rec_learner(model, dls)
    learn.fine_tune(epochs, lr, freeze_epochs=freeze_epochs)
    save_recognizer(model, out or ROOT / "models" / "rec_finetuned.npz")
    return learn


## Command


In [ ]:
#| export
#| notest
def main() -> None:
    parser = argparse.ArgumentParser(description="Fine-tune the PP-OCRv6 medium recognizer with fastai")
    parser.add_argument("labels", type=Path, help="label file: image path, a tab, then the text")
    parser.add_argument("--root", type=Path, help="folder the image paths start from; defaults to the label file's folder")
    parser.add_argument("--init", type=Path, help="starting ONNX or .npz weights; defaults to the official checkpoint")
    parser.add_argument("--out", type=Path, help="where to write the .npz; defaults to models/rec_finetuned.npz")
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--freeze-epochs", type=int, default=1)
    parser.add_argument("--bs", type=int, default=16)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--valid-pct", type=float, default=0.1)
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    finetune(args.labels, root=args.root, init=args.init, out=args.out, epochs=args.epochs,
             freeze_epochs=args.freeze_epochs, bs=args.bs, lr=args.lr, valid_pct=args.valid_pct,
             num_workers=args.workers, seed=args.seed)


In [ ]:
#| export
#| notest
#| eval: false
if __name__ == "__main__":
    main()


## Try it on synthetic lines


Real data is yours to bring. To check the loop end to end there is a generator that renders random text lines with Pillow's built-in font and writes them with a label file.


In [ ]:
#| export
#| notest
def synthetic_lines(root: Path, n: int, seed: int = 0, alphabet: str = "abcdefghijklmnopqrstuvwxyz0123456789") -> Path:
    "Render `n` short text lines under `root` and return the label file."
    from PIL import Image, ImageDraw, ImageFont

    rng = random.Random(seed)
    root = Path(root)
    (root / "img").mkdir(parents=True, exist_ok=True)
    rows = []
    for index in range(n):
        text = "".join(rng.choice(alphabet) for _ in range(rng.randint(3, 12)))
        font = ImageFont.load_default(size=rng.randint(24, 40))
        left, top, right, bottom = font.getbbox(text)
        image = Image.new("RGB", (right + 24, bottom + 16), (255, 255, 255))
        ImageDraw.Draw(image).text((12, 8 - top // 2), text, fill=(rng.randint(0, 60),) * 3, font=font)
        name = f"img/{index:05d}.png"
        image.save(root / name)
        rows.append(f"{name}\t{text}")
    labels = root / "labels.txt"
    labels.write_text("\n".join(rows) + "\n", encoding="utf-8")
    return labels


In [ ]:
#| notest
import tempfile

tmp = Path(tempfile.mkdtemp())
labels = synthetic_lines(tmp, 6)
items = read_labels(labels)
test_eq(len(items), 6)
charset = load_charset(ROOT / "models" / "ppocrv6_dict.txt", add_space=True)
dls = rec_dataloaders(items, charset, bs=4, valid_pct=0.34)
images, targets = next(iter(dls.train))
test_eq(tuple(images.shape[1:]), (3, 48, 320))
test_eq(images.shape[0], targets.shape[0])


The official model already reads clean print, so fine-tuning on these lines as they are changes nothing. Instead the test relabels every line in upper case. The images still show lower case, so the model has to change what it outputs. Before training the character error rate is high. After a few head-only epochs it has to fall.


In [ ]:
#| notest
labels = synthetic_lines(tmp / "train", 96)
rows = [line.partition("\t") for line in labels.read_text().splitlines()]
upper = labels.with_name("upper.txt")
upper.write_text("".join(f"{name}\t{text.upper()}\n" for name, _, text in rows))
test_eq(upper.read_text().splitlines()[0].split("\t")[1].isupper(), True)


In [ ]:
#| notest
official = ROOT / "models" / "medium_rec.onnx"
dls = rec_dataloaders(read_labels(upper), charset, bs=16, valid_pct=0.2)
before = rec_learner(load_recognizer(official), dls).validate()[1]

out = tmp / "rec.npz"
learn = finetune(upper, out=out, epochs=0, freeze_epochs=6, bs=16, lr=3e-3, valid_pct=0.2)
after = learn.validate()[1]
print(f"cer before {before:.3f}, after {after:.3f}")
test_eq(before > 0.3, True)
test_eq(after < 0.05, True)


The saved weights must give the MAX graph the same output as the trained PyTorch model. Both are compared on one validation image.


In [ ]:
#| notest
from solid_lens.model import compile_recognizer

trained = load_recognizer(out).eval()
x = next(iter(learn.dls.valid))[0][:1]
with torch.no_grad():
    ours = trained.probabilities(x).numpy()
tuned = compile_recognizer(cache_dir=ROOT / "models" / "native_medium_rec_48x320.mef", checkpoint=out)
theirs = tuned.execute(np.ascontiguousarray(x.numpy()))[0].to_numpy()
print("max abs diff", float(np.abs(ours - theirs).max()))
test_eq(bool(np.allclose(ours, theirs, atol=1e-4)), True)
test_eq(ours.argmax(-1).tolist(), theirs.argmax(-1).tolist())
